# ANGSI JR. V8.3 — JupyterLab Launcher
Run one cell at a time. This notebook is included in the ZIP.


In [ ]:
from pathlib import Path
import sys, os, subprocess, py_compile, urllib.request, time
APP_DIR = Path("/data/epic20/APGCE/apgce_team_13/ANGSI_JR_V8_3_FULL")
APP = APP_DIR / "angsi_jr_v8.py"
assert APP.is_file(), f"Missing: {APP}"
py_compile.compile(str(APP), doraise=True)
print("App syntax: PASS; location:", APP)


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(APP_DIR / "requirements.txt")])
print("Dependencies installed in notebook Python:",sys.executable)


In [ ]:
env = os.environ.copy()
env["ANGSI_DATA_ROOT"] = "/data/epic20/APGCE/input_data/challenge3"
LOG = APP_DIR / "server_v83.log"
with LOG.open("a") as f:
    process = subprocess.Popen([sys.executable,"-m","streamlit","run",str(APP),"--server.address=127.0.0.1","--server.port=8512","--server.headless=true","--server.fileWatcherType=none"],cwd=str(APP_DIR),env=env,stdout=f,stderr=subprocess.STDOUT,start_new_session=True)
print("Launched PID", process.pid, "at port 8512. Log:", LOG)


In [ ]:
for _ in range(12):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8512/_stcore/health",timeout=2) as res:
            print("Streamlit health",res.status,res.read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    print(LOG.read_text(errors="replace")[-5000:])
print("Open http://127.0.0.1:8512 only on the SAME remote HPC desktop, or configured JupyterHub proxy")
